In [1]:
import numpy as np
import meshio
import os
import matplotlib.pyplot as plt

In [7]:
def generate_Ex(potential, delta_x):
    Ex = np.ndarray(shape = (potential.shape[0] - 1, potential.shape[1] - 1))

    for i in range(potential.shape[0] - 1):
        for j in range(potential.shape[1] - 1):
            Ex[i][j] = (potential[i][j] - potential[i][j + 1]) / delta_x

    return Ex

def generate_Ey(potential, delta_y):
    Ey = np.ndarray(shape = (potential.shape[0] - 1, potential.shape[1] - 1))

    for i in range(potential.shape[0] - 1):
        for j in range(potential.shape[1] - 1):
            Ey[i][j] = (potential[i][j] - potential[i + 1][j]) / delta_y

    return Ey

def generate_pseudo_potential(Ex, Ey, Q, M, Freq):
    return (Ex ** 2 + Ey ** 2) * Q ** 2 / (4 * M * Freq ** 2)

def convert_vtu_to_csv(potential_vtu_file_path,
                       potential_file_path, 
                       pp_file_path, 
                       Ex_file_path, 
                       Ey_file_path, 
                       Ez_file_path,
                       Q, M, Freq):
    
    mesh = meshio.read(potential_vtu_file_path)

    x_set = sorted(set(mesh.points[:, 0]))
    y_set = sorted(set(mesh.points[:, 1]))
    u = mesh.point_data['u']

    potential = np.ndarray(shape = (len(x_set), len(y_set)))
    for i in range(len(x_set)):
        for j in range(len(y_set)):
            potential[i][j] = u[len(y_set) * i + j]

    Ex = generate_Ex(potential, x_set[1] - x_set[0])
    Ey = generate_Ey(potential, y_set[1] - y_set[0])
    pp = generate_pseudo_potential(Ex, Ey, Q, M, Freq)

    np.savetxt(potential_file_path, potential, delimiter=',')
    np.savetxt(pp_file_path, pp, delimiter=',')
    np.savetxt(Ex_file_path, Ex, delimiter=',')
    np.savetxt(Ey_file_path, Ey, delimiter=',')
    np.savetxt(Ez_file_path, np.zeros(shape = Ex.shape), delimiter=',')

In [9]:
convert_vtu_to_csv("/Users/alekseidushanin/Documents/GitHub/LFI_KT/miniproject/paraview_data/potential000000.vtu",
                  "/Users/alekseidushanin/Documents/GitHub/LFI_KT/miniproject/dataset/potential.csv",
                  "/Users/alekseidushanin/Documents/GitHub/LFI_KT/miniproject/dataset/pp.csv",
                  "/Users/alekseidushanin/Documents/GitHub/LFI_KT/miniproject/dataset/Ex.csv",
                  "/Users/alekseidushanin/Documents/GitHub/LFI_KT/miniproject/dataset/Ey.csv",
                  "/Users/alekseidushanin/Documents/GitHub/LFI_KT/miniproject/dataset/Ez.csv",
                  1, 1, 1)

In [59]:
class Ion_Trap_Simulation:
    def __init__(self,
                 input_file_path, 
                 cpp_script_path, 
                 potential_vtu_output_path,
                 potential_file_path, 
                 pp_file_path, 
                 Ex_file_path, 
                 Ey_file_path, 
                 Ez_file_path,
                 electrode_positions, 
                 electrode_widths,
                 electrode_voltages,
                 Q, M, Freq):
        self.input_file_path = input_file_path
        self.cpp_script_path = cpp_script_path
        self.potential_vtu_output_path = potential_vtu_output_path
        self.potential_file_path = potential_file_path
        self.pp_file_path = pp_file_path
        self.Ex_file_path = Ex_file_path
        self.Ey_file_path = Ey_file_path 
        self.Ez_file_path = Ez_file_path
        self.electrode_positions = electrode_positions
        self.electrode_widths = electrode_widths
        self.electrode_voltages = electrode_voltages
        self.Q = Q
        self.M = M
        self.Freq = Freq

    def init_input_file(self):
        #Clear the input file
        with open(self.input_file_path, 'w'):
            pass
        
        #Fill the input file
        f = open(self.input_file_path, 'w')
        f.write(self.potential_vtu_output_path + '\n')
        
        [f.write(str(i) + '\n') for i in self.electrode_positions]
        f.write('\n')

        [f.write(str(i) + '\n') for i in self.electrode_widths]
        f.write('\n')

        [f.write(str(i) + '\n') for i in self.electrode_voltages]
        
        f.close()

    def perform_simulation(self):
        #Run c++ sim script (get's start parameters from *self.input_file_path*)
        os.system(self.cpp_script_path)
        
        #Convert .vtu to .csv
        convert_vtu_to_csv(self.potential_vtu_output_path,
                          self.potential_file_path,
                          self.pp_file_path,
                          self.Ex_file_path,
                          self.Ey_file_path,
                          self.Ez_file_path,
                          self.Q, 
                          self.M,
                          self.Freq)

In [68]:
sim = Ion_Trap_Simulation('/Users/alekseidushanin/Documents/GitHub/LFI_KT/miniproject/dataset/sim_input.txt',
                         '/Users/alekseidushanin/Documents/GitHub/LFI_KT/miniproject/test',
                         '/Users/alekseidushanin/Documents/GitHub/LFI_KT/miniproject/dataset/output1.vtk',
                         '', 
                         '', 
                         '', 
                         '', 
                         '',
                         [0, 0.2, 0.4, 0.6, 0.8], 
                         [0.2, 0.2, 0.2, 0.2, 0.2],
                         [0, 100, 0, 100, 0],
                         1, 2, 3)

sim.init_input_file()